# Feature Engineering

## Objective

In this notebook, we will turn the cleaned dataset into a **model-ready feature set**.

The focus is not on creating a large number of features. Instead, we will create features that are:

- useful for predicting the target
- available at prediction time
- free from data leakage
- consistent with the cleaned data
- suitable for machine learning
- easy to explain

## Target

Our target variable is:

`company_response_to_consumer`

This is a **multiclass classification** problem because the target contains multiple response categories.

## Feature Engineering Process

We will follow these steps:

1. Load the cleaned dataset
2. Confirm the target and selected features
3. Remove columns that should not be used for modeling
4. Create useful date/time features
5. Create relevant numerical features
6. Encode categorical features
7. Process text features where useful
8. Check for data leakage
9. Check the final feature matrix
10. Prepare the dataset for modeling

All features will be created using information that would be available **at the time of prediction**.

The final output of this notebook will be a clean, consistent, and **model-ready feature set** for the next stage: **Modeling**.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

In [3]:
file_path = "/content/drive/MyDrive/Consumer Complain Project/data/processed/complaints_cleaned.parquet"
df = pd.read_parquet(file_path)
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (90112, 16)


,date_received,product,sub_product,issue,sub_issue,consumer_complaint_narrative,company_public_response,company,state,zip_code,tags,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response,complaint_id
0,2026-04-25 20:09:10+00:00,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Investigation took more than 30 days,None,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",WA,98087,No tag,Web,2026-04-25 20:17:54+00:00,Closed with explanation,Yes,21603527
1,2026-02-14 21:18:34+00:00,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,None,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",AL,35022,No tag,Web,2026-02-14 21:19:00+00:00,Closed with explanation,Yes,19508852
2,2026-05-25 18:41:22+00:00,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Their investigation did not fix an error on yo...,None,None,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,75287,No tag,Web,2026-05-25 18:45:42+00:00,None,Yes,22543000
3,2026-05-12 15:06:42+00:00,Debt collection,I do not know,Communication tactics,"You told them to stop contacting you, but they...",None,None,Western Management Consultants,NY,146XX,No tag,Web,2026-05-12 15:12:25+00:00,Closed with explanation,Yes,22116348
4,2026-05-01 08:25:14+00:00,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,None,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,SC,29455,No tag,Web,2026-05-01 08:25:37+00:00,Closed with explanation,Yes,21783248


## 1. Target Scoping

Per the EDA and Feature Audit decisions: drop `timely_response` (leakage), `company_public_response` (largely redundant with the target and mostly missing), `date_sent_to_company` (used earlier for forwarding-time analysis but not part of the confirmed feature set), and `complaint_id` (an identifier, not a feature).

In [4]:
target = "company_response_to_consumer"

leakage_and_id_columns = [
    "complaint_id",
    "company_public_response",
    "timely_response",
    "date_sent_to_company"
]

print("Target:", target)
print("\nColumns dropped for leakage/identifier reasons:")
print(leakage_and_id_columns)

Target: company_response_to_consumer

Columns dropped for leakage/identifier reasons:
['complaint_id', 'company_public_response', 'timely_response', 'date_sent_to_company']


In [5]:
print("Target distribution:")
print(df[target].value_counts(dropna=False))

print("\nTarget missing percentage:", f"{df[target].isna().mean() * 100:.2f}%")

Target distribution:
company_response_to_consumer
Closed with explanation            50849
None                               15807
Closed with non-monetary relief    11897
In progress                         9467
Closed with monetary relief         1576
Untimely response                    516
Name: count, dtype: int64

Target missing percentage: 17.54%


### Key Finding — Target Scoping

Matches EDA exactly: `Closed with explanation` dominates, `In progress` and missing values together account for roughly 28% of rows and will be excluded below since neither represents a known, final outcome.

In [6]:
model_df = df[
    df[target].notna() &
    (df[target] != "In progress")
].copy()

print("Original rows:", len(df))
print("Modeling rows:", len(model_df))
print("Rows removed:", len(df) - len(model_df))

print("\nTarget distribution after scoping:")
print(model_df[target].value_counts())

Original rows: 90112
Modeling rows: 64838
Rows removed: 25274

Target distribution after scoping:
company_response_to_consumer
Closed with explanation            50849
Closed with non-monetary relief    11897
Closed with monetary relief         1576
Untimely response                    516
Name: count, dtype: int64


In [8]:
assert model_df[target].isna().sum() == 0, "Missing target values still present"
assert (model_df[target] == "In progress").sum() == 0, "In progress rows still present"
print("Scoping check passed: no missing or In progress rows remain.")

Scoping check passed: no missing or In progress rows remain.


## 2. Feature-Level Missingness

The Feature Audit already confirmed the six core features (`product`, `sub_product`, `issue`, `company`, `submitted_via`, `state`) have 0% missing values. Re-checking here on the scoped modeling population, and including the columns we're about to engineer from (`date_received`, `consumer_complaint_narrative`).

In [7]:
candidate_columns = [
    "product", "sub_product", "issue", "sub_issue", "company",
    "state", "zip_code", "tags", "submitted_via",
    "date_received", "consumer_complaint_narrative"
]

print(model_df[candidate_columns].isna().sum().sort_values(ascending=False))

consumer_complaint_narrative    45667
sub_product                         0
product                             0
issue                               0
sub_issue                           0
state                               0
company                             0
zip_code                            0
tags                                0
submitted_via                       0
date_received                       0
dtype: int64


### Key Finding — Feature-Level Missingness

The six audited features remain fully populated. `tags` and `consumer_complaint_narrative` carry the missingness seen in EDA (94.8% and ~78% respectively) — handled below: `tags` is dropped entirely (see Section 4), and narrative missingness is handled by design, since `narrative_present` is built specifically to capture whether it's missing.

## 3. Engineered Features — Date and Narrative

**Note on scope:** the six features validated in EDA and the Feature Audit are all categorical. The features below (date components, narrative-derived signals) were not part of that validation — they're added here as a reasonable, common-practice extension, not because EDA proved they matter. They're kept for two reasons: they're cheap to compute, and a tree-based model can simply ignore them if they carry no signal. If feature importance in modeling shows they're dead weight, they should be dropped there.

One caution worth flagging now rather than discovering later: `received_hour` assumes `date_received` carries a genuine time-of-day value reflecting when the complaint was actually submitted. If this timestamp instead reflects when the record was processed or loaded into the system, `received_hour` would be a weak proxy for a data-pipeline artifact rather than real complainant behavior. Worth a quick sanity check against the CFPB data dictionary before trusting this feature's importance in modeling.

In [9]:
model_df["received_year"] = model_df["date_received"].dt.year
model_df["received_month"] = model_df["date_received"].dt.month
model_df["received_dayofweek"] = model_df["date_received"].dt.dayofweek
model_df["received_day"] = model_df["date_received"].dt.day
model_df["received_quarter"] = model_df["date_received"].dt.quarter
model_df["received_hour"] = model_df["date_received"].dt.hour

print(
    model_df[
        [
            "date_received",
            "received_year",
            "received_month",
            "received_dayofweek",
            "received_day",
            "received_quarter",
            "received_hour"
        ]
    ].head()
)

              date_received  received_year  received_month  \
0 2026-04-25 20:09:10+00:00           2026               4   
1 2026-02-14 21:18:34+00:00           2026               2   
3 2026-05-12 15:06:42+00:00           2026               5   
4 2026-05-01 08:25:14+00:00           2026               5   
7 2026-03-05 01:07:05+00:00           2026               3   

   received_dayofweek  received_day  received_quarter  received_hour  
0                   5            25                 2             20  
1                   5            14                 1             21  
3                   1            12                 2             15  
4                   4             1                 2              8  
7                   3             5                 1              1  


In [10]:
date_features = [
    "received_year",
    "received_month",
    "received_dayofweek",
    "received_day",
    "received_quarter",
    "received_hour"
]

print(model_df[date_features].describe().T)

                      count         mean       std     min     25%     50%  \
received_year       64838.0  2025.988633  0.188860  2014.0  2026.0  2026.0   
received_month      64838.0     4.367315  1.241247     1.0     4.0     4.0   
received_dayofweek  64838.0     2.331364  1.770112     0.0     1.0     2.0   
received_day        64838.0    16.446559  8.656801     1.0     9.0    16.0   
received_quarter    64838.0     1.781409  0.457468     1.0     2.0     2.0   
received_hour       64838.0    13.623122  6.925592     0.0     8.0    15.0   

                       75%     max  
received_year       2026.0  2026.0  
received_month         5.0    12.0  
received_dayofweek     4.0     6.0  
received_day          24.0    31.0  
received_quarter       2.0     4.0  
received_hour         19.0    23.0  


In [11]:
narrative = model_df["consumer_complaint_narrative"].fillna("").astype(str)

model_df["narrative_present"] = narrative.str.strip().ne("").astype(int)
model_df["narrative_length"] = narrative.str.len()
model_df["narrative_word_count"] = narrative.str.split().str.len()

print("Narrative present:")
print(model_df["narrative_present"].value_counts())

print("\nNarrative length / word count summary:")
print(model_df[["narrative_length", "narrative_word_count"]].describe().T)

Narrative present:
narrative_present
0    45667
1    19171
Name: count, dtype: int64

Narrative length / word count summary:
                        count        mean         std  min  25%  50%    75%  \
narrative_length      64838.0  401.371912  997.984570  0.0  0.0  0.0  355.0   
narrative_word_count  64838.0   66.466702  163.062425  0.0  0.0  0.0   63.0   

                          max  
narrative_length      31553.0  
narrative_word_count   5205.0  


### Key Finding — Engineered Features

received_hour shows a plausible daily pattern — low overnight (0-7am: under 2,000
each), rising through the day, peaking in early evening (16-20h: 4,300-4,500), then
declining. This looks like real submission-time behavior, not a flat/random artifact,
so it's reasonable to trust as a genuine feature rather than a data-pipeline proxy.

narrative_present shows 19,171 of 64,838 rows (29.6%) have a narrative — slightly
higher than the 22.3% figure from the original full-dataset EDA, since this population
excludes In Progress/missing-target rows, which may skew narrative availability
slightly. Not a concern, just worth noting the denominator changed.

## 4. Drop Columns Not Carried Forward

`zip_code` is dropped — too high-cardinality to be useful without far more granular geographic modeling, and `state` already captures the geographic signal EDA found. `tags` is dropped — 94.8% missing, and it was never part of the feature set validated in EDA or the Feature Audit. Raw `date_received` and `consumer_complaint_narrative` are dropped now that their engineered versions exist.

In [12]:
columns_to_drop = [
    "zip_code",
    "tags",
    "date_received",
    "consumer_complaint_narrative"
] + leakage_and_id_columns

model_df = model_df.drop(columns=columns_to_drop)

print("Remaining columns:")
print(model_df.columns.tolist())

Remaining columns:
['product', 'sub_product', 'issue', 'sub_issue', 'company', 'state', 'submitted_via', 'company_response_to_consumer', 'received_year', 'received_month', 'received_dayofweek', 'received_day', 'received_quarter', 'received_hour', 'narrative_present', 'narrative_length', 'narrative_word_count']


## 5. Flagging 'Pending Company Match'

The Feature Audit flagged `"Pending Company Match"` as very likely a placeholder for complaints where CFPB's company-matching process failed — not a real company. Left as-is, frequency encoding would treat it as if it were one company's actual response pattern, which it isn't. Relabeling it to an explicit `"Unknown Company"` value is a fixed, rule-based change (not something learned from data distribution), so it's safe to apply before the train/test split — unlike category bucketing, this introduces no leakage.

In [13]:
pending_count = (model_df["company"] == "Pending Company Match").sum()
print("Pending Company Match rows before relabeling:", pending_count)

model_df["company"] = model_df["company"].replace(
    "Pending Company Match", "Unknown Company"
)

print("Confirmed relabeled:", (model_df["company"] == "Unknown Company").sum())

Pending Company Match rows before relabeling: 1
Confirmed relabeled: 1


## 6. Train/Test Split

The split happens here, before any frequency- or threshold-based transformation. Everything after this point that learns from category counts — rare-category bucketing and company frequency encoding — is fit on `X_train` only, then applied unchanged to `X_test`. This is the fix for the leakage issue found in the previous version of this notebook, where bucketing was computed on the full dataset before splitting.

In [14]:
from sklearn.model_selection import train_test_split

X = model_df.drop(columns=[target])
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(4) * 100)

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(4) * 100)

Training shape: (51870, 16)
Test shape: (12968, 16)

Training target distribution:
company_response_to_consumer
Closed with explanation            78.42
Closed with non-monetary relief    18.35
Closed with monetary relief         2.43
Untimely response                   0.80
Name: proportion, dtype: float64

Test target distribution:
company_response_to_consumer
Closed with explanation            78.42
Closed with non-monetary relief    18.35
Closed with monetary relief         2.43
Untimely response                   0.79
Name: proportion, dtype: float64


## 7. Rare-Category Bucketing — Fit on Training Data Only

Per the Feature Audit spec, `sub_product`, `issue`, `sub_issue`, and `state` all get a Top-N + "Other" treatment at a 30-complaint threshold. The category list to keep is determined from `X_train` alone, then applied identically to `X_test` — a category considered "rare" is decided using only information the model would have at training time.

In [15]:
def bucket_rare_categories(train_series, test_series, min_count=30):
    counts_in_train = train_series.value_counts()
    categories_to_keep = counts_in_train[counts_in_train >= min_count].index

    train_bucketed = train_series.where(train_series.isin(categories_to_keep), "Other")
    test_bucketed = test_series.where(test_series.isin(categories_to_keep), "Other")

    return train_bucketed, test_bucketed

In [16]:
columns_to_bucket = ["sub_product", "issue", "sub_issue", "state"]

for column in columns_to_bucket:
    X_train[column], X_test[column] = bucket_rare_categories(
        X_train[column], X_test[column], min_count=30
    )
    print(f"{column}: {X_train[column].nunique()} categories in train, "
          f"{X_test[column].nunique()} categories in test after bucketing")

sub_product: 38 categories in train, 38 categories in test after bucketing
issue: 59 categories in train, 59 categories in test after bucketing
sub_issue: 90 categories in train, 90 categories in test after bucketing
state: 50 categories in train, 50 categories in test after bucketing


### Key Finding — Bucketing

sub_product dropped from 58 to 38 categories, state from 60 to 50 — both meaningfully
reduced, fixing the gap from the previous version where these went straight into
one-hot encoding unbucketed. issue (92→59) and sub_issue (206→90) also bucketed
cleanly. Zero unseen categories appeared in the test set for any bucketed column,
confirming the "Other" bucket is doing its job.

## 8. Company Frequency Encoding

`company` (1,499 raw categories, including the now-relabeled `"Unknown Company"`) is too high-cardinality for one-hot or Top-N bucketing to be practical. Frequency encoding — replacing each company with how often it appears in the training data — captures company size without exploding dimensionality. Fit on `X_train` only; unseen companies in `X_test` get a frequency of 0.

In [17]:
company_frequency = X_train["company"].value_counts()

X_train["company_frequency"] = X_train["company"].map(company_frequency).fillna(0)
X_test["company_frequency"] = X_test["company"].map(company_frequency).fillna(0)

print("Training company frequency summary:")
print(X_train["company_frequency"].describe())

unseen_in_test = (X_test["company_frequency"] == 0).sum()
print("\nCompanies in test with zero training frequency (unseen):", unseen_in_test)

Training company frequency summary:
count    51870.000000
mean      7365.170812
std       5222.556384
min          1.000000
25%        423.000000
50%      10992.000000
75%      11135.000000
max      11490.000000
Name: company_frequency, dtype: float64

Companies in test with zero training frequency (unseen): 115


In [18]:
X_train = X_train.drop(columns=["company"])
X_test = X_test.drop(columns=["company"])

print("company column dropped after encoding. Remaining shape:", X_train.shape)

company column dropped after encoding. Remaining shape: (51870, 16)


## 9. Unseen-Category Check Before One-Hot Encoding

Before fitting the encoder, checking which remaining categorical columns have values in `X_test` that never appeared in `X_train`. This isn't a problem to fix — `OneHotEncoder(handle_unknown="ignore")` handles it safely — but it's worth seeing up front rather than assuming it away.

In [19]:
categorical_features = X_train.select_dtypes(include="object").columns.tolist()
print("Categorical features going into one-hot encoding:")
print(categorical_features)

for column in categorical_features:
    train_categories = set(X_train[column].unique())
    test_categories = set(X_test[column].unique())
    unseen = test_categories - train_categories
    print(f"{column}: {len(unseen)} unseen test categories")

Categorical features going into one-hot encoding:
['product', 'sub_product', 'issue', 'sub_issue', 'state', 'submitted_via']
product: 0 unseen test categories
sub_product: 0 unseen test categories
issue: 0 unseen test categories
sub_issue: 0 unseen test categories
state: 0 unseen test categories
submitted_via: 0 unseen test categories


## 10. One-Hot Encoding

Fit on `X_train` only; `handle_unknown="ignore"` means any unseen test category simply gets all-zero indicator columns rather than raising an error.

In [20]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_encoded = encoder.fit_transform(X_train[categorical_features])
X_test_encoded = encoder.transform(X_test[categorical_features])

print("Encoded training shape:", X_train_encoded.shape)
print("Encoded test shape:", X_test_encoded.shape)
print("Encoded feature count:", len(encoder.get_feature_names_out()))

Encoded training shape: (51870, 252)
Encoded test shape: (12968, 252)
Encoded feature count: 252


## 11. Assemble the Final Feature Matrix

Combine the numeric features (date components, narrative features, company frequency) with the one-hot encoded categorical block into a single, model-ready DataFrame.

In [21]:
numeric_features = [
    "received_year", "received_month", "received_dayofweek",
    "received_day", "received_quarter", "received_hour",
    "narrative_present", "narrative_length", "narrative_word_count",
    "company_frequency"
]

X_train_final = pd.DataFrame(
    X_train[numeric_features].to_numpy(),
    columns=numeric_features,
    index=X_train.index
).join(
    pd.DataFrame(
        X_train_encoded,
        columns=encoder.get_feature_names_out(),
        index=X_train.index
    )
)

X_test_final = pd.DataFrame(
    X_test[numeric_features].to_numpy(),
    columns=numeric_features,
    index=X_test.index
).join(
    pd.DataFrame(
        X_test_encoded,
        columns=encoder.get_feature_names_out(),
        index=X_test.index
    )
)

print("Final training shape:", X_train_final.shape)
print("Final test shape:", X_test_final.shape)

Final training shape: (51870, 262)
Final test shape: (12968, 262)


## 12. Final Quality Checks

Before saving, confirm there are no missing or infinite values, and that train/test columns match exactly — a mismatch here would silently break modeling later.

In [22]:
print("Missing values — training:", X_train_final.isna().sum().sum())
print("Missing values — test:", X_test_final.isna().sum().sum())

print("\nInfinite values — training:", np.isinf(X_train_final.to_numpy()).sum())
print("Infinite values — test:", np.isinf(X_test_final.to_numpy()).sum())

print("\nColumns match exactly:", list(X_train_final.columns) == list(X_test_final.columns))

print("\nTraining data types:")
print(X_train_final.dtypes.value_counts())

Missing values — training: 0
Missing values — test: 0

Infinite values — training: 0
Infinite values — test: 0

Columns match exactly: True

Training data types:
float64    252
int64       10
Name: count, dtype: int64


In [23]:
print("Final feature engineering audit")
print("-" * 40)
print("Training rows:", X_train_final.shape[0])
print("Test rows:", X_test_final.shape[0])
print("Total features:", X_train_final.shape[1])
print("\nFeature groups:")
print("Numeric features:", len(numeric_features))
print("One-hot encoded features:", len(encoder.get_feature_names_out()))
print("\nTarget class distribution (train):")
print(y_train.value_counts())

Final feature engineering audit
----------------------------------------
Training rows: 51870
Test rows: 12968
Total features: 262

Feature groups:
Numeric features: 10
One-hot encoded features: 252

Target class distribution (train):
company_response_to_consumer
Closed with explanation            40679
Closed with non-monetary relief     9517
Closed with monetary relief         1261
Untimely response                    413
Name: count, dtype: int64


## 13. Save Model-Ready Datasets

In [24]:
processed_path = "/content/drive/MyDrive/Consumer Complain Project/data/processed"

X_train_final.to_parquet(f"{processed_path}/X_train.parquet", index=True)
X_test_final.to_parquet(f"{processed_path}/X_test.parquet", index=True)
y_train.to_frame(name=target).to_parquet(f"{processed_path}/y_train.parquet", index=True)
y_test.to_frame(name=target).to_parquet(f"{processed_path}/y_test.parquet", index=True)

print("Model-ready datasets saved successfully.")

Model-ready datasets saved successfully.


In [25]:
X_train_check = pd.read_parquet(f"{processed_path}/X_train.parquet")
X_test_check = pd.read_parquet(f"{processed_path}/X_test.parquet")
y_train_check = pd.read_parquet(f"{processed_path}/y_train.parquet")
y_test_check = pd.read_parquet(f"{processed_path}/y_test.parquet")

print("X_train:", X_train_check.shape)
print("X_test:", X_test_check.shape)
print("y_train:", y_train_check.shape)
print("y_test:", y_test_check.shape)

print("\nFeature columns match:", list(X_train_check.columns) == list(X_test_check.columns))

X_train: (51870, 262)
X_test: (12968, 262)
y_train: (51870, 1)
y_test: (12968, 1)

Feature columns match: True


## Final Summary

Feature engineering is complete, and the dataset is ready for modeling.

### Modeling Population

64,838 of 90,112 total rows retained (25,274 excluded: 15,807 missing target, 9,467
In Progress) — an unresolved-outcome exclusion rate of ~28%, consistent with EDA.

### Leakage Prevention

- `timely_response`, `company_public_response`, `date_sent_to_company`, and `complaint_id` were dropped as either leakage risks or identifiers.
- Rare-category bucketing (`sub_product`, `issue`, `sub_issue`, `state`) was fit on the training split only, then applied to test — fixing the leakage present in the previous version of this notebook.
- Company frequency encoding was fit on the training split only.
- `"Pending Company Match"` was relabeled to `"Unknown Company"` before encoding, per the Feature Audit's recommendation, rather than being treated as a real company.

### Features Created

262 total features: 10 numeric (6 date components, 3 narrative-derived, 1 company
frequency) + 252 one-hot encoded categorical features. One limitation worth flagging
for modeling: 115 companies in the test set had zero training frequency, meaning
those rows carry no company-specific signal.

### Scope Note

Date-derived and narrative-derived features were added as a reasonable extension beyond the six features validated in EDA/Feature Audit — not because they were independently validated. Review their feature importance in modeling before assuming they matter.

### Next Step

Proceed to **Modeling** — establish a baseline and compare multiclass classifiers using metrics that account for class imbalance (macro-F1, per-class recall) rather than accuracy alone.